<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [5]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber PyPDF2

Cloning into 'Lettura_bilanci'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 41 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 29.83 MiB | 18.36 MiB/s, done.
Resolving deltas: 100% (11/11), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 86.4 MB/s eta 0:00:00


In [28]:
import PyPDF2
import pdfplumber
import pandas as pd

pdf_path_2= "Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf"

with open(pdf_path_2, "rb") as file:
    reader = PyPDF2.PdfReader(file)  # Read the PDF
    text = ""
    for page in reader.pages:
        text += page.extract_text() + ""

print(text[:100])

with pdfplumber.open(pdf_path_2) as pdf:
    page = pdf.pages[2]
    tables = page.extract_tables()

    if tables:
        t = tables[0]
        print(t.shape)
        if t:
            print("KO")
        t_clean = [row for row in t if any(cell is not None for cell in row)]
        max_cols = max(len(row) for row in t_clean)
        t_uniform = [row + [None] * (max_cols - len(row)) for row in t_clean]

        df= pd.DataFrame(t_uniform)
        print(df.shape)
        print(df)




In [13]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali",
                                                 "crediti",
                                                 "immobilizzazioni materiali", "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


pdf_path_2= "Lettura_bilanci/deposito bilanci/IFB SRL - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)

                                                    0           1           2
0                                                      31-12-2024  31-12-2023
1                                  Stato patrimoniale                        
2                                              Attivo                        
3   A) Crediti verso soci per versamenti ancora do...           0           0
4                                 B) Immobilizzazioni                        
..                                                ...         ...         ...
80  20) Imposte sul reddito dell'esercizio, corren...                        
81                                   imposte correnti      23.095           0
82                     imposte differite e anticipate       6.775     (6.554)
83  Totale delle imposte sul reddito dell'esercizi...      29.870     (6.554)
84                 21) Utile (perdita) dell'esercizio   2.509.948    (56.242)

[85 rows x 3 columns]


In [14]:
import glob
import os
pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

risultati = {}
summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path)
    if tabella is not None:
        if tabella.shape[1] == 3:
          subset = tabella.iloc[[0], [1,2]].loc["imposte correnti"]
          print(subset)



KeyError: 'imposte correnti'

In [12]:
print(risultati["IFB SRL - 2024"])

KeyError: 'IFB SRL - 2024'